# Asynchronous Advantage Actor-Critic (A3C) for Atari Kung Fu Master Using Deep Reinforcement Learning

This project implements an Asynchronous Advantage Actor-Critic (A3C) reinforcement learning agent capable of learning to play the Atari game Kung Fu Master using deep neural networks and parallelized environment training. The model is built with PyTorch and trained using multiple asynchronous agents interacting with the Gymnasium Atari environment simultaneously. Through convolutional feature extraction, policy optimization, and value estimation, the agent learns effective gameplay strategies directly from raw pixel observations.

The gymnasium documentation can be found [here](https://ale.farama.org/environments/kung_fu_master/).

## Overview of A3C

Asynchronous Advantage Actor-Critic (A3C) is a reinforcement learning algorithm that improves training efficiency and stability by running multiple agents in parallel across independent environments. Unlike Deep Q-Learning, A3C directly learns both a policy function and a value function.

The algorithm consists of two primary components:

- **Actor Network:** Learns which actions to take.
- **Critic Network:** Estimates how good the current state is.

Multiple agents explore different parts of the environment simultaneously and asynchronously update a shared global network. This parallel training strategy improves exploration diversity, reduces training correlation, and accelerates learning.

## Installing the Required Packages and Importing the Libraries

### Installing Gymnasium

In [ ]:
!pip install gymnasium
!pip install "gymnasium[atari, accept-rom-license]"
!pip install ale-py
!apt-get install -y swig
!pip install gymnasium[box2d]

### Importing the Libraries

In [ ]:
import cv2
import math
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch.multiprocessing as mp
import torch.distributions as distributions
from torch.distributions import Categorical
import ale_py
import gymnasium as gym
from gymnasium.spaces import Box
from gymnasium import ObservationWrapper

## Building the AI

### Creating the Architecture of the Neural Network

The project uses a Convolutional Neural Network (CNN) architecture to process Atari game frames.

**Convolutional Layers** - Extract spatial features from visual game frames; Detect movement patterns, enemies, player position, and obstacles; Reduce image dimensionality while preserving important information

**Fully Connected Layers** - Transform extracted visual features into higher-level representations; Feed information into policy and value estimation heads

**Actor Output** - Produces action probabilities for selecting the next move; Uses a probability distribution for exploration

**Critic Output** - Estimates the expected future reward (state value); Helps stabilize policy updates during training

In [ ]:
class Network(nn.Module):
  def __init__(self, action_size):
    super(Network, self).__init__()
    self.conv1 = nn.Conv2d(in_channels = 4, out_channels = 32, kernel_size = (3,3), stride = 2)   # 4 corresponds to 4 grayscale frames
    self.conv2 = nn.Conv2d(in_channels = 32, out_channels = 32, kernel_size = (3,3), stride = 2)
    self.conv3 = nn.Conv2d(in_channels = 32, out_channels = 32, kernel_size = (3,3), stride = 2)
    self.flatten = nn.Flatten()   # flattening layer
    self.fc1 = nn.Linear(in_features = 32 * 4 * 4, out_features = 128)
    self.fc2a = nn.Linear(in_features = 128, out_features = action_size)  # output layer 1 - action Q-values
    self.fc2s = nn.Linear(in_features = 128, out_features = 1)  # output layer 2 - state value


  # forward propagation
  def forward(self, state):
    x = self.conv1(state)
    x = F.relu(x)
    x = self.conv2(x)
    x = F.relu(x)
    x = self.conv3(x)
    x = F.relu(x)
    x = self.flatten(x)
    x = self.fc1(x)
    x = F.relu(x)
    action_values = self.fc2a(x)
    state_value = self.fc2s(x)[0]   # just the state value
    return action_values, state_value

## Training the AI

### Setting Up the Environment

In [ ]:
class PreprocessAtari(ObservationWrapper):

  def __init__(self, env, height = 42, width = 42, crop = lambda img: img, dim_order = 'pytorch', color = False, n_frames = 4):
    super(PreprocessAtari, self).__init__(env)
    self.img_size = (height, width)
    self.crop = crop
    self.dim_order = dim_order
    self.color = color
    self.frame_stack = n_frames
    n_channels = 3 * n_frames if color else n_frames
    obs_shape = {'tensorflow': (height, width, n_channels), 'pytorch': (n_channels, height, width)}[dim_order]
    self.observation_space = Box(0.0, 1.0, obs_shape)
    self.frames = np.zeros(obs_shape, dtype = np.float32)

  def reset(self):
    self.frames = np.zeros_like(self.frames)
    obs, info = self.env.reset()
    self.update_buffer(obs)
    return self.frames, info

  def observation(self, img):
    img = self.crop(img)
    img = cv2.resize(img, self.img_size)
    if not self.color:
      if len(img.shape) == 3 and img.shape[2] == 3:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    img = img.astype('float32') / 255.
    if self.color:
      self.frames = np.roll(self.frames, shift = -3, axis = 0)
    else:
      self.frames = np.roll(self.frames, shift = -1, axis = 0)
    if self.color:
      self.frames[-3:] = img
    else:
      self.frames[-1] = img
    return self.frames

  def update_buffer(self, obs):
    self.frames = self.observation(obs)

def make_env():
  env = gym.make('KungFuMasterNoFrameskip-v0', render_mode = 'rgb_array')
  env = PreprocessAtari(env, height = 42, width = 42, crop = lambda img: img, dim_order = 'pytorch', color = False, n_frames = 4)
  return env

env = make_env()

state_shape = env.observation_space.shape
number_actions = env.action_space.n
print("State shape:", state_shape)
print("Number actions:", number_actions)
print("Action names:", env.env.env.env.get_action_meanings())

State shape: (4, 42, 42)
Number actions: 14
Action names: ['NOOP', 'UP', 'RIGHT', 'LEFT', 'DOWN', 'DOWNRIGHT', 'DOWNLEFT', 'RIGHTFIRE', 'LEFTFIRE', 'DOWNFIRE', 'UPRIGHTFIRE', 'UPLEFTFIRE', 'DOWNRIGHTFIRE', 'DOWNLEFTFIRE']


/usr/local/lib/python3.12/dist-packages/gymnasium/envs/registration.py:513: DeprecationWarning: WARN: The environment KungFuMasterNoFrameskip-v0 is out of date. You should consider upgrading to version `v4`.
  logger.deprecation(


### Initializing the Hyperparameters

**Learning Rate** - Controls how much the network updates after each optimization step

**Discount Factor (Gamma)** - Determines the importance of future rewards

**Number of Environments** - Number of asynchronous agents training simultaneously

In [ ]:
learning_rate = 1e-4
discount_factor = 0.99
number_environments = 10    # multiple agents in multiple envrionments are trained

### Implementing the A3C Class

In [ ]:
class Agent():
  def __init__(self, action_size):
    self.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    self.action_size = action_size
    self.network = Network(action_size).to(self.device)
    self.optimizer = optim.Adam(self.network.parameters(), lr = learning_rate)


  # agent plays an action given a certain state with a softmax policy
  def act(self, state):
    if state.ndim == 3:   # if the state is not in a batch (single observation)
      state = [state]   # state now in a batch
    state = torch.tensor(state, dtype = torch.float32, device = self.device)    # convert to tensors
    action_values, _ = self.network.forward(state)    # get action values (disregarding the state_value for now)

    # implementing softmax policy
    policy = F.softmax(action_values, dim = -1)   # converts action_values to probability, applying softmax to the last dimension representing different actions
    return np.array([np.random.choice(len(p), p = p) for p in policy.detach().cpu().numpy()])   # return selected action using sampling (several actions corresponding to each state in the batch)


  # updates the model's parameter (agent learns)
  def step(self, state, action, reward, next_state, done):
    batch_size = state.shape[0]   # get batch size (state is a tensor)

    # convert to torch tensors
    state = torch.tensor(state, dtype = torch.float32, device = self.device)
    next_state = torch.tensor(next_state, dtype = torch.float32, device = self.device)
    reward = torch.tensor(reward, dtype = torch.float32, device = self.device)
    done = torch.tensor(done, dtype = torch.bool, device = self.device).to(dtype = torch.float32)   # convert to float

    action_values, state_value = self.network(state)    # get action values and state value
    _, next_state_value = self.network(next_state)    # next state value (we don't need action values)
    target_state_value = reward + discount_factor * next_state_value * (1 - done)   # compute target state value using bellman equation
    advantage = target_state_value - state_value    # compute advantage for A3C model

    # compute critic loss and actor loss
    # actor loss: compute entropy using the distribution of probabilities and log probabilities over the action values
    probs = F.softmax(action_values, dim = -1)
    logprobs = F.log_softmax(action_values, dim = -1)
    entropy = -torch.sum(probs * logprobs, axis = -1)

    batch_idx = np.arange(batch_size)   # to get the log probabilies of the actions actually selected (action selected from whole batch of states)
    logp_actions = logprobs[batch_idx, action]

    actor_loss = -(logp_actions * advantage.detach()).mean() - 0.001 * entropy.mean()   # exploration feature at the end to balance importance of entropy
    critic_loss = F.mse_loss(target_state_value.detach(), state_value)
    total_loss = actor_loss + critic_loss   # get total loss

    # backpropagate loss and update weight using optimizer
    self.optimizer.zero_grad()    # reset optimizer
    total_loss.backward()    # backpropagate loss
    self.optimizer.step()    # update weights


### Initializing the A3C Agent

In [ ]:
agent = Agent(number_actions)

### Evaluating our A3C Agent on a Single Episode

In [ ]:
# returns a list of total rewards in each of the episodes
def evaluate(agent, env, n_episodes=1):
  episodes_rewards = []
  for _ in range(n_episodes):
    state, _ = env.reset()    # initialize the state
    total_reward = 0
    while True:   # until an episode is done
      action = agent.act(state)    # play an action in order to reach the next state
      state, reward, done, info, _ = env.step(action[0])    # get the next state, reward, and done; info is some other things
      total_reward += reward
      if done:
        break
    # once the episode is done
    episodes_rewards.append(total_reward)
  return episodes_rewards

### Testing Multiple Agents on Multiple Environments at the Same Time

A3C uses multiple worker agents running in parallel environments. Each worker interacts with its own environment independently while sharing a common global neural network. Each worker periodically updates the shared global model using gradients computed from its local experiences.

In [ ]:
# implementing the asynchronous part of A3C (multiple agents interacting with multiple environments)
class EnvBatch:
  def __init__(self, n_envs=10):
    self.envs = [make_env() for _ in range(n_envs)]    # creating a list of multiple environments


  # reset multiple envs at the same time
  def reset(self):
    # creating new list of initialized states
    _states = []
    for env in self.envs:
      _states.append(env.reset()[0])
    return np.array(_states)


  # step into multiple environments simultaneously
  def step(self, actions):
    # step method call from multiple env returns next_states, rewards, dones, infos
    next_states, rewards, dones, infos, _ = map(np.array, zip(*[env.step(a) for env, a in zip(self.envs, actions)]))    # transform single step method into a multiple stepping method using loop
    # check is any of these envs have finished (i.e., done = true)
    for i in range(len(self.envs)):
      if dones[i]:
        next_states[i] = self.envs[i].reset()[0]   # reset the state of that env
    return next_states, rewards, dones, infos

### Training the A3C Agent

In [ ]:
import tqdm   # displays progress bar

env_batch = EnvBatch(number_environments)   # instace of EnvBatch class (multiple envs)
batch_states = env_batch.reset()    # reset all states in these multiple envs

# training loop
with tqdm.trange(0, 3001) as progress_bar:
  for i in progress_bar:    # same as i in range(0, 3001)
    batch_actions = agent.act(batch_states)   # play an action on the batch of states initialized in multiple envs
    batch_next_states, batch_rewards, batch_dones, _ = env_batch.step(batch_actions)    # get the next states, rewards, and dones

    # call step method of the agent class which is how the agents actually learn
    batch_rewards *= 0.01   # since we're dealing with high rewards, we reduce the maginitude of these rewards to stabalize training
    agent.step(batch_states, batch_actions, batch_rewards, batch_next_states, batch_dones)

    # update the batch_states; if we don't we will still be in the initialized batch_states
    batch_states = batch_next_states

    # print average accumulated reward every 1000 iterations of the agent over 10 episodes
    if i % 1000 == 0:
      print("Average agent reward: ", np.mean(evaluate(agent, env, n_episodes=10)))

/usr/local/lib/python3.12/dist-packages/gymnasium/envs/registration.py:513: DeprecationWarning: WARN: The environment KungFuMasterNoFrameskip-v0 is out of date. You should consider upgrading to version `v4`.
  logger.deprecation(
  0%|          | 0/3001 [00:00<?, ?it/s]/tmp/ipykernel_2204/1361483384.py:46: UserWarning: Using a target size (torch.Size([1])) that is different to the input size (torch.Size([10])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  critic_loss = F.mse_loss(target_state_value.detach(), state_value)
/tmp/ipykernel_2204/1361483384.py:13: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  state = torch.tensor(state, dtype = torch.float32, device = self.device)    # convert to tensors
  0%|       

Average agent reward:  730.0


 34%|███▎      | 1012/3001 [04:10<1:35:51,  2.89s/it]

Average agent reward:  1230.0


 67%|██████▋   | 2011/3001 [06:20<40:59,  2.48s/it]

Average agent reward:  790.0


100%|██████████| 3001/3001 [08:27<00:00,  5.91it/s]

Average agent reward:  720.0


## Visualizing the Results

In [ ]:
import glob
import io
import base64
import imageio
from IPython.display import HTML, display

def show_video_of_model(agent, env):
  state, _ = env.reset()
  done = False
  frames = []
  while not done:
    frame = env.render()
    frames.append(frame)
    action = agent.act(state)
    state, reward, done, _, _ = env.step(action[0])
  env.close()
  imageio.mimsave('video.mp4', frames, fps=30)

show_video_of_model(agent, env)

def show_video():
    mp4list = glob.glob('*.mp4')
    if len(mp4list) > 0:
        mp4 = mp4list[0]
        video = io.open(mp4, 'r+b').read()
        encoded = base64.b64encode(video)
        display(HTML(data='''<video alt="test" autoplay
                loop controls style="height: 400px;">
                <source src="data:video/mp4;base64,{0}" type="video/mp4" />
             </video>'''.format(encoded.decode('ascii'))))
    else:
        print("Could not find video")

show_video()